# Synthetic Tau-typer walkthrough
All data and model sequences are artificial. This tests software mechanics only. See README.md for assumptions. Do not train a new model in this walkthrough.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pySuStaIn as sus
D = Path.cwd()
if not (D / 'df_sustain_true.csv').exists():
    D = Path('/Users/ro0681ag/Documents/ChatGPT/Jake Papers 3/data_simulate')
gt = pd.read_csv(D / 'df_sustain_true.csv')
gt.study.value_counts()

## Load and align the reference data
These filenames match the old notebook but contain only synthetic values.

In [ ]:
orig = pd.read_csv(D / 'X_merge_preExp3.csv', index_col=0)
mod_cols = np.load(D / 'mod_cols.npy', allow_pickle=False)
mod_ids = np.load(D / 'mod_ids.npy', allow_pickle=False)
mt_inferred = pd.read_csv(D / 'inferred.csv', index_col=0)
base = pd.read_csv(D / 'Cons2_Base_z.csv', index_col=0)
base = base.loc[mt_inferred.index, mod_cols]
base['subtype'] = mt_inferred.subtype
base['stage'] = mt_inferred.stage
orig.index = orig.sid.values
orig = orig.loc[base.index]
for frame in (orig, gt):
    frame['Choroid_tot'] = frame[[c for c in frame if 'horoid' in c]].mean(axis=1)
orig.shape, base.shape

## Aggregate and normalize
Use the fixed artificial baseline rather than refitting mixture models. Choroid is retained for format testing but is not a simulated covariate effect.

In [ ]:
L_regs = json.loads((D / 'region_definitions.json').read_text())
Lobes = list(L_regs)
base_L = pd.DataFrame({b: base[cols].mean(axis=1) for b, cols in L_regs.items()})
orig_L = pd.DataFrame({b: orig[cols].mean(axis=1) for b, cols in L_regs.items()})
gt_L = pd.DataFrame({b: gt[cols].mean(axis=1) for b, cols in L_regs.items()})
normal = json.loads((D / 'normalization.json').read_text())
W_gt = pd.DataFrame({b: (gt_L[b] - normal[b]['mean']) / normal[b]['sd'] for b in Lobes})
W_gt.index = gt.filename
truth = pd.read_csv(D / 'new_data_truth.csv', index_col=0)
assert np.allclose(W_gt.values, truth[['observed_z_' + b for b in Lobes]].values)
W_gt.head()

## Apply the fixed stand-in model
No training. Twenty repeated copies of each sequence support the sampling API; they do not encode posterior uncertainty.

In [ ]:
cfg = json.loads((D / 'config.json').read_text())
indata = pd.read_csv(D / 'MTBFTplus_input.csv', index_col=0)
with np.load(D / 'model_samples.npz', allow_pickle=False) as saved:
    samples_sequence = saved['samples_sequence']
    samples_f = saved['samples_f']
import tempfile
with tempfile.TemporaryDirectory() as scratch:
    model = sus.ZscoreSustain(indata[Lobes].values, np.array(cfg['Z_vals']), np.array(cfg['Z_max']), Lobes, 1, 4, 1, scratch, 'SYNTHETIC', False)
    results = model.subtype_and_stage_individuals_newData(
        data_new=W_gt[Lobes].values, samples_sequence=samples_sequence,
        samples_f=samples_f, N_samples=20)
ml_subtype, prob_ml_subtype, ml_stage, prob_ml_stage, prob_subtype, prob_stage, prob_subtype_stage = results
predicted_stage = ml_stage.ravel().astype(int)
predicted_subtype = ml_subtype.ravel().astype(int) + 1
predicted_subtype[predicted_stage == 0] = 0
result = pd.DataFrame({'subtype': predicted_subtype, 'stage': predicted_stage, 'p_subtype': prob_ml_subtype.ravel(), 'p_stage': prob_ml_stage.ravel()}, index=W_gt.index)
result.head()

## Compare predictions with generation labels
These are different: noisy measurements and overlapping trajectories prevent perfect recovery. Separately, expected_predictions.csv provides the reproducible software output.

In [ ]:
positive = truth.true_stage > 0
print('Subtype accuracy among generated stage > 0:', (result.loc[positive, 'subtype'] == truth.loc[positive, 'true_subtype']).mean())
print('Stage mean absolute error:', (result.stage - truth.true_stage).abs().mean())
expected = pd.read_csv(D / 'expected_predictions.csv', index_col=0)
assert np.array_equal(result.subtype.values, expected.subtype.values)
assert np.array_equal(result.stage.values, expected.stage.values)
result.to_csv(D / 'walkthrough_predictions.csv')